# What drives cell typing and imputation: XGBoost + TreeSHAP (Jackson, Danenberg, Keren)

**Questions** (`future_plan.md` §18.1):
1. Which inputs drive each coarse cell type: the cell's own markers, its neighbours' markers, or nuclear content / size / density?
   Does this differ between Jackson and Danenberg?
2. When a Jackson-trained model types Danenberg cells, which features push the wrong class?
3. Do own × neighbour interactions carry the endothelial gain (the panel has no vessel marker)?
4. Imputation: which inputs predict SMA in each cohort (it flips sign across cohorts)? Which inputs predict H3K27me3 in Jackson
   (14 shared channels) and in Keren (all channels: is phospho-S6, the mass neighbour of H3K27me3, dominant)?

**Data and features:** the same cells, coarse labels, replicate-0 split and Table + nbr features as `cell-type-graph`.
- Features: own 14 channels + log area (cohort-scaled), plus the kNN-10 neighbours' mean / max / SD / 2-hop mean of the same,
  ring-30 µm count, contact count and mean neighbour distance.

**Models:** XGBoost (`hist`, 300 rounds, depth 6, η 0.1, class-weighted), trained on Jackson, Danenberg or both. Macro-F1 is
reported next to `cell-type-graph`'s LightGBM, to confirm the model is comparable.

**SHAP:**
- TreeSHAP (`pred_contribs`) on 20,000 random test cells per model × test cohort.
- Per class: mean |SHAP| over the cells of that true class, for each feature and summed per feature group.
  Groups = source (own / neighbour) × marker family (epithelial, hormone/HER2, immune, stroma, proliferation, nuclear, size/density).
- Interactions (`pred_interactions`) on 2,000 cells: own × neighbour interaction mass per class.
- **Check against SHAP credit-splitting between correlated markers:** grouped permutation importance (drop in macro-F1 when a
  group's columns are permuted together), on the same test cells.

Descriptive analysis: no decision depends on it.

In [ ]:
import os, glob, json, time, sys, subprocess, warnings, gc, traceback
import numpy as np, pandas as pd
import pyarrow.parquet as pq
import scipy.sparse as sp
from scipy.spatial import cKDTree
from scipy import stats
try:
    import xgboost as xgb
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost"], check=True)
    import xgboost as xgb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

IN = os.environ.get("IN_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
SMOKE = os.environ.get("SMOKE") == "1" or os.path.exists(f"{IN}/SMOKE")
SEED = 20261013
MIN_CELLS = 200
COF = 5.0
N_FIT = 20_000 if SMOKE else 300_000
N_SHAP = 2_000 if SMOKE else 20_000
N_INT = 300 if SMOKE else 2_000
SMOKE_IMAGES = 40
XGB = dict(n_estimators=40 if SMOKE else 300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
           tree_method="hist", n_jobs=os.cpu_count(), random_state=SEED)
T0 = time.time()
FAILED = []

NUC = ["histone_h3", "dna1", "dna2"]
MARKERS = ["c_erb_b_2_her2", "cd20", "cd3", "cd45", "cd68", "cytokeratin_5", "cytokeratin_8_18",
           "estrogen_receptor_alpha", "ki_67", "pan_cytokeratin", "sma"]
FEATS = NUC + MARKERS
D_TABLE = {"c_erb_b_2_her2": "c_erb_b_2_her2_3b5"}
COARSE = ["epithelial", "T cell", "B cell", "myeloid", "endothelial", "stroma"]
D_COARSE = {"CD8^{+} T cells": "T cell", "CD4^{+} T cells": "T cell", "T_{Reg} & T_{Ex}": "T cell",
            "B cells": "B cell",
            "Macrophages": "myeloid", "Granulocytes": "myeloid", "Macrophages & granulocytes": "myeloid",
            "Endothelial": "endothelial",
            "Fibroblasts": "stroma", "Fibroblasts FSP1^{+}": "stroma", "Myofibroblasts": "stroma",
            "Myofibroblasts PDPN^{+}": "stroma"}
J_COARSE = {1: "T cell", 2: "B cell", 4: "myeloid", 6: "myeloid", 7: "endothelial",
            **{m: "stroma" for m in range(8, 14)}, **{m: "epithelial" for m in range(14, 28)}}
LABEL = {"histone_h3": "H3", "dna1": "DNA1", "dna2": "DNA2", "c_erb_b_2_her2": "HER2", "cd20": "CD20", "cd3": "CD3",
         "cd45": "CD45", "cd68": "CD68", "cytokeratin_5": "CK5", "cytokeratin_8_18": "CK8/18", "estrogen_receptor_alpha": "ERα",
         "ki_67": "Ki-67", "pan_cytokeratin": "panCK", "sma": "SMA", "log_area": "area"}
FAMILY = {"pan_cytokeratin": "epithelial", "cytokeratin_8_18": "epithelial", "cytokeratin_5": "epithelial",
          "estrogen_receptor_alpha": "hormone/HER2", "c_erb_b_2_her2": "hormone/HER2", "cd45": "immune", "cd3": "immune",
          "cd20": "immune", "cd68": "immune", "sma": "stroma", "ki_67": "proliferation", "histone_h3": "nuclear",
          "dna1": "nuclear", "dna2": "nuclear", "log_area": "size/density"}

def tick(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

def g(pattern):
    return sorted(glob.glob(f"{IN}/**/{pattern}", recursive=True))

def near(name, anchor):
    hits = g(name)
    assert hits, f"missing input {name}"
    return max(hits, key=lambda c: len(os.path.commonpath([c, anchor])))
print("SMOKE" if SMOKE else "FULL", "| xgboost", xgb.__version__)

## Cells, labels and Table + nbr features (as in `cell-type-graph`)

In [ ]:
P = {}
for p in g("cells.parquet"):
    names = pq.read_schema(p).names
    if "object_number" not in names or "published__h3k27me3" in names:
        continue
    key = "Danenberg" if "metabric_id" in names else "Jackson"
    P[key] = dict(cells=p, contacts=near("cell_neighbours.parquet", p))
P["Danenberg"]["split"] = near("danenberg_patient_split.csv", P["Danenberg"]["cells"])
print(json.dumps(P, indent=1))


def load(key):
    cells = pd.read_parquet(P[key]["cells"])
    if key == "Danenberg":
        cells = cells[(cells.is_tumour == 1) & (cells.is_hot_aggregate == 0)]
        pid = "metabric_id"
    else:
        cells = cells[cells.is_tumour == 1]
        pid = "patient_id"
    n_per = cells.groupby("image_name").size()
    keep = n_per.index[n_per >= MIN_CELLS]
    if SMOKE:
        keep = np.sort(np.random.default_rng(SEED).choice(np.asarray(keep), SMOKE_IMAGES, replace=False))
    cells = cells[cells.image_name.isin(keep)].sort_values(["image_name", "object_number"]).reset_index(drop=True)
    n = len(cells)
    img, img_names = pd.factorize(cells.image_name, sort=True)
    starts = np.r_[0, np.flatnonzero(np.diff(img)) + 1, n]
    loc = pd.Series(np.arange(n), index=pd.MultiIndex.from_arrays([cells.image_number.values, cells.object_number.values]))
    con = pd.read_parquet(P[key]["contacts"])
    ca = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["from"].values])).values
    cb = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["to"].values])).values
    ok = ~(np.isnan(ca) | np.isnan(cb))
    ca, cb = ca[ok].astype(np.int64), cb[ok].astype(np.int64)
    CON = sp.coo_matrix((np.ones(int((ca != cb).sum())), (ca[ca != cb], cb[ca != cb])), shape=(n, n)).tocsr()
    CON = ((CON + CON.T) > 0).astype(np.float64).tocsr()
    cols = [D_TABLE.get(c, c) for c in FEATS] if key == "Danenberg" else FEATS
    X = np.c_[np.arcsinh(cells[cols].values.astype(np.float64) / COF),
              np.log(np.maximum(cells["area"].values, 1))].astype(np.float32)
    phen = cells.cell_phenotype.astype(str).values
    fine = sorted(set(phen))
    y_fine = pd.Series(phen).map({p: i for i, p in enumerate(fine)}).values.astype(np.int64)
    if key == "Danenberg":
        epi = cells.groupby(cells.cell_phenotype.astype(str)).is_epithelial.mean()
        cmap = {p: ("epithelial" if epi[p] > 0.5 else D_COARSE.get(p)) for p in fine}
    else:
        mc = cells.groupby(cells.cell_phenotype.astype(str)).metacluster.first()
        cmap = {p: J_COARSE.get(int(mc[p])) for p in fine}
    coarse_of = np.array([COARSE.index(cmap[p]) if cmap[p] else -1 for p in fine], np.int64)
    return dict(key=key, n=n, starts=starts, n_img=len(img_names), img_names=np.asarray(img_names),
                obj=cells.object_number.values.astype(np.int64), patient=cells[pid].astype(str).values,
                xy=cells[["x", "y"]].values.astype(np.float64), CON=CON, X=X, fine=fine,
                y={"fine": y_fine, "coarse": coarse_of[y_fine]})

CO = {k: load(k) for k in ("Jackson", "Danenberg")}
CJ, CD = CO["Jackson"], CO["Danenberg"]
split0 = pd.read_csv(P["Danenberg"]["split"]).set_index("metabric_id").split
CD["split0"] = pd.Series(CD["patient"]).map(split0).fillna("none").values

def harm(X):
    med = np.nanmedian(X, 0)
    q05, q95 = np.nanpercentile(X, [5, 95], 0)
    sc = np.where(q95 - q05 > 1e-3, q95 - q05, np.nanstd(X, 0) + 1e-6)
    return ((X - med) / sc).astype(np.float32)

def knn10(C, k=10):
    rows, cols = [], []
    for a, b in zip(C["starts"][:-1], C["starts"][1:]):
        kk = min(k, b - a - 1)
        _, nb = cKDTree(C["xy"][a:b]).query(C["xy"][a:b], k=kk + 1)
        rows.append(np.repeat(np.arange(a, b), kk)); cols.append(nb[:, 1:].ravel() + a)
    A = sp.coo_matrix((np.ones(sum(map(len, rows)), np.float32), (np.concatenate(rows), np.concatenate(cols))),
                      shape=(C["n"], C["n"])).tocsr()
    A = ((A + A.T) > 0).astype(np.float32).tocsr()
    A.sort_indices()
    return A

def r30_count(C):
    out = np.zeros(C["n"], np.int64)
    for a, b in zip(C["starts"][:-1], C["starts"][1:]):
        out[a:b] = cKDTree(C["xy"][a:b]).query_ball_point(C["xy"][a:b], 30.0, return_length=True) - 1
    return out

def nbr_features(C, A):
    X = C["XS"]["cohort"]
    deg = np.diff(A.indptr).astype(np.float32)[:, None]
    mean = (A @ X) / deg
    std = np.sqrt(np.maximum((A @ (X * X)) / deg - mean * mean, 0))
    mx = np.maximum.reduceat(X[A.indices], A.indptr[:-1], axis=0)
    hop2 = (A @ mean) / deg
    dst = np.repeat(np.arange(C["n"]), np.diff(A.indptr))
    dist = np.bincount(dst, np.linalg.norm(C["xy"][A.indices] - C["xy"][dst], axis=1), C["n"]) / deg.ravel()
    return np.c_[X, mean, mx, std, hop2, np.log1p(r30_count(C)), np.log1p(np.diff(C["CON"].indptr)), dist / 30.0].astype(np.float32)

for key, C in CO.items():
    C["XS"] = {"cohort": np.nan_to_num(harm(C["X"]))}
    C["XS"]["nbr"] = nbr_features(C, knn10(C))
    print(f"{key}: {C['n']:,} cells, {C['n_img']} images, {len(set(C['patient']))} patients, {len(C['fine'])} fine labels")
tick("tables, labels and neighbour features")

In [ ]:
D_TRVA_PAT = np.unique(CD["patient"][CD["split0"] != "test"])

def split_images(C, cell_split):
    s = cell_split[C["starts"][:-1]]
    return {k: np.flatnonzero(s == k) for k in ("train", "val", "test")}

def make_splits(REP):
    jp, jcode = np.unique(CJ["patient"], return_inverse=True)
    u = np.random.default_rng(SEED + 11 + REP).random(len(jp))
    J_SPLIT = np.where(u < 0.7, "train", np.where(u < 0.8, "val", "test"))[jcode]
    uv = np.random.default_rng(SEED + 21 + REP).random(len(D_TRVA_PAT))
    dval = set(D_TRVA_PAT[uv < 0.125])
    D_SPLIT = np.where(CD["split0"] == "test", "test", np.where(pd.Series(CD["patient"]).isin(dval).values, "val", "train"))
    IMG = {"Jackson": split_images(CJ, J_SPLIT), "Danenberg": split_images(CD, D_SPLIT)}
    IMG["Both"] = {s: np.r_[IMG["Jackson"][s], IMG["Danenberg"][s] + CJ["n_img"]] for s in ("train", "val")}
    return IMG

def cells_of(C, imgs):
    imgs = np.asarray(imgs, np.int64)
    if not len(imgs):
        return np.zeros(0, np.int64)
    return np.concatenate([np.arange(C["starts"][i], C["starts"][i + 1]) for i in imgs])

IMG0 = make_splits(0)
for c in ("Jackson", "Danenberg"):
    print(c, {s: len(v) for s, v in IMG0[c].items()}, "images")

In [ ]:
BASE = FEATS + ["log_area"]
FNAMES = ([f"own:{c}" for c in BASE] + [f"nbr_mean:{c}" for c in BASE] + [f"nbr_max:{c}" for c in BASE] +
          [f"nbr_sd:{c}" for c in BASE] + [f"nbr_hop2:{c}" for c in BASE] +
          ["density:ring30_count", "density:contact_count", "density:mean_nbr_distance"])
assert len(FNAMES) == CJ["XS"]["nbr"].shape[1], (len(FNAMES), CJ["XS"]["nbr"].shape)

def group_of(f):
    src, ch = f.split(":")
    if src == "density":
        return "neighbours: size/density"
    return f"{'own' if src == 'own' else 'neighbours'}: {FAMILY[ch]}"
GROUP = np.array([group_of(f) for f in FNAMES])
GROUPS = sorted(set(GROUP), key=lambda s: (not s.startswith("own"), s))
print(pd.Series(GROUP).value_counts().to_string())

## XGBoost cell-type models (coarse labels) and their scores

In [ ]:
def combine(A, B):
    return dict(key="Both", n=A["n"] + B["n"], n_img=A["n_img"] + B["n_img"], starts=np.r_[A["starts"][:-1], B["starts"] + A["n"]],
                patient=np.r_[A["patient"], B["patient"]], cohort=np.r_[np.zeros(A["n"], np.int8), np.ones(B["n"], np.int8)],
                XS={"nbr": np.r_[A["XS"]["nbr"], B["XS"]["nbr"]]}, y={"coarse": np.r_[A["y"]["coarse"], B["y"]["coarse"]]})

def class_weights(y, ncls):
    c = np.bincount(y, minlength=ncls).astype(float)
    return np.where(c > 0, c.sum() / (ncls * np.maximum(c, 1)), 0.0)

def train_rows(C, imgs, seed):
    idx = cells_of(C, imgs)
    idx = idx[C["y"]["coarse"][idx] >= 0]
    rs = np.random.default_rng(seed)
    pools = [idx[C["cohort"][idx] == c] for c in (0, 1)] if C["key"] == "Both" else [idx]
    return np.sort(np.concatenate([rs.choice(p, min(N_FIT, len(p)), replace=False) for p in pools if len(p)]))

def macro_f1(y, p, k=len(COARSE)):
    cm = np.bincount(y * k + p, minlength=k * k).reshape(k, k).astype(float)
    tp, sup, prd = np.diag(cm), cm.sum(1), cm.sum(0)
    f1 = np.where(sup + prd > 0, 2 * tp / np.maximum(sup + prd, 1), 0)
    return float(f1[sup > 0].mean()), f1

IMG = IMG0
MODELS, TEST, SCORES = {}, {}, []
for t in ("Jackson", "Danenberg"):
    node = cells_of(CO[t], IMG[t]["test"])
    TEST[t] = node[CO[t]["y"]["coarse"][node] >= 0]
for src in ("Jackson", "Danenberg", "Both"):
    C = combine(CJ, CD) if src == "Both" else CO[src]
    fit = train_rows(C, IMG[src]["train"], SEED)
    y = C["y"]["coarse"][fit]
    mdl = xgb.XGBClassifier(**XGB)
    mdl.fit(C["XS"]["nbr"][fit], y, sample_weight=class_weights(y, len(COARSE))[y])
    MODELS[src] = mdl
    for t in ("Jackson", "Danenberg"):
        X_, y_ = CO[t]["XS"]["nbr"][TEST[t]], CO[t]["y"]["coarse"][TEST[t]]
        f1, f1c = macro_f1(y_, mdl.predict(X_))
        SCORES.append(dict(train=src, test=t, macro_f1=f1, **{f"f1_{c}": v for c, v in zip(COARSE, f1c)}))
    del C
    gc.collect()
    tick(f"XGBoost trained on {src}")
SC = pd.DataFrame(SCORES)
SC.to_csv(f"{OUT}/xgb_scores.csv", index=False)
print(SC.round(3).to_string(index=False))

## TreeSHAP per class and feature group

In [ ]:
rs = np.random.default_rng(SEED + 1)
SHAP_ROWS, SAMPLE = [], {}
for t in ("Jackson", "Danenberg"):
    SAMPLE[t] = np.sort(rs.choice(TEST[t], min(N_SHAP, len(TEST[t])), replace=False))

def contribs(mdl, X):
    """(n, n_class, n_feat + 1) TreeSHAP values in margin (log-odds) space; the last column is the bias."""
    v = mdl.get_booster().predict(xgb.DMatrix(X), pred_contribs=True)
    return v.reshape(len(X), len(COARSE), -1)

SH = {}
for src in ("Jackson", "Danenberg", "Both"):
    for t in ("Jackson", "Danenberg"):
        X_ = CO[t]["XS"]["nbr"][SAMPLE[t]]
        y_ = CO[t]["y"]["coarse"][SAMPLE[t]]
        p_ = MODELS[src].predict(X_)
        V = contribs(MODELS[src], X_)[:, :, :-1]
        SH[(src, t)] = (V, y_, p_)
        for c, cname in enumerate(COARSE):
            m = y_ == c
            if not m.any():
                continue
            a = np.abs(V[m, c, :]).mean(0)
            s_ = V[m, c, :].mean(0)
            for j, f in enumerate(FNAMES):
                SHAP_ROWS.append(dict(train=src, test=t, cls=cname, feature=f, group=GROUP[j], mean_abs_shap=float(a[j]),
                                      mean_shap=float(s_[j]), n_cells=int(m.sum())))
        tick(f"SHAP {src} -> {t}")
SHD = pd.DataFrame(SHAP_ROWS)
SHD.to_csv(f"{OUT}/shap_by_feature.csv", index=False)
GRP = SHD.groupby(["train", "test", "cls", "group"]).mean_abs_shap.sum().reset_index()
GRP["share"] = GRP.mean_abs_shap / GRP.groupby(["train", "test", "cls"]).mean_abs_shap.transform("sum")
GRP.to_csv(f"{OUT}/shap_by_group.csv", index=False)
with pd.option_context("display.width", 250, "display.max_rows", 400):
    for src, t in (("Jackson", "Jackson"), ("Danenberg", "Danenberg"), ("Jackson", "Danenberg")):
        print(f"--- share of |SHAP| by group, trained on {src}, tested on {t}")
        print(GRP[(GRP.train == src) & (GRP.test == t)].pivot_table(index="group", columns="cls", values="share")
              .reindex(GROUPS)[COARSE].round(3).to_string())
        print(f"--- top 6 features per class, trained on {src}, tested on {t}")
        top = SHD[(SHD.train == src) & (SHD.test == t)].sort_values("mean_abs_shap", ascending=False).groupby("cls").head(6)
        for c in COARSE:
            tt = top[top.cls == c]
            print(f"{c:12s}", ", ".join(f"{r.feature} {r.mean_abs_shap:.2f}" for r in tt.itertuples()))

## Cohort agreement of the drivers, and what misleads the Jackson model on Danenberg

In [ ]:
AGREE = []
for c in COARSE:
    a = SHD[(SHD.train == "Jackson") & (SHD.test == "Jackson") & (SHD.cls == c)].set_index("feature").mean_abs_shap
    b = SHD[(SHD.train == "Danenberg") & (SHD.test == "Danenberg") & (SHD.cls == c)].set_index("feature").mean_abs_shap
    if len(a) and len(b):
        common = a.index.intersection(b.index)
        rho = stats.spearmanr(a[common], b[common]).correlation
        top_a, top_b = set(a.nlargest(10).index), set(b.nlargest(10).index)
        AGREE.append(dict(cls=c, spearman_feature_importance=rho, top10_overlap=len(top_a & top_b)))
AGREE = pd.DataFrame(AGREE)
AGREE.to_csv(f"{OUT}/driver_agreement_j_vs_d.csv", index=False)
print(AGREE.round(3).to_string(index=False))

# misclassified Danenberg cells under the Jackson model: features pushing toward the wrong (predicted) class
V, y_, p_ = SH[("Jackson", "Danenberg")]
MIS = []
cm = pd.crosstab(pd.Series(y_).map(dict(enumerate(COARSE))), pd.Series(p_).map(dict(enumerate(COARSE))))
cm.to_csv(f"{OUT}/confusion_j_to_d_sample.csv")
print(cm.to_string())
for (tc, pc_), n in pd.Series(list(zip(y_, p_))).value_counts().items():
    if tc == pc_ or n < 50:
        continue
    m = (y_ == tc) & (p_ == pc_)
    push = V[m, pc_, :] - V[m, tc, :]                                   # toward the wrong class, against the true one
    mp = push.mean(0)
    for j in np.argsort(-mp)[:8]:
        MIS.append(dict(true=COARSE[tc], predicted=COARSE[pc_], n_cells=int(n), feature=FNAMES[j], group=GROUP[j],
                        mean_push=float(mp[j])))
MISD = pd.DataFrame(MIS)
MISD.to_csv(f"{OUT}/misclassification_drivers_j_to_d.csv", index=False)
with pd.option_context("display.width", 250):
    print(MISD.groupby(["true", "predicted"]).head(4).round(3).to_string(index=False))

## Own × neighbour interactions (Jackson model, Jackson test cells)

In [ ]:
try:
    sub = SAMPLE["Jackson"][:N_INT]
    I = MODELS["Jackson"].get_booster().predict(xgb.DMatrix(CJ["XS"]["nbr"][sub]), pred_interactions=True)
    I = I.reshape(len(sub), len(COARSE), len(FNAMES) + 1, len(FNAMES) + 1)[:, :, :-1, :-1]
    own = np.array([f.startswith("own:") for f in FNAMES])
    y_ = CJ["y"]["coarse"][sub]
    INT = []
    for c, cname in enumerate(COARSE):
        m = y_ == c
        if not m.any():
            continue
        A = np.abs(I[m, c]).mean(0)
        main = np.diag(A).copy()
        np.fill_diagonal(A, 0)
        INT.append(dict(cls=cname, main_own=float(main[own].sum()), main_nbr=float(main[~own].sum()),
                        inter_own_own=float(A[np.ix_(own, own)].sum()), inter_own_nbr=float(2 * A[np.ix_(own, ~own)].sum()),
                        inter_nbr_nbr=float(A[np.ix_(~own, ~own)].sum()), n_cells=int(m.sum())))
    INT = pd.DataFrame(INT)
    tot = INT[["main_own", "main_nbr", "inter_own_own", "inter_own_nbr", "inter_nbr_nbr"]].sum(1)
    for col in ("main_own", "main_nbr", "inter_own_own", "inter_own_nbr", "inter_nbr_nbr"):
        INT[f"{col}_share"] = INT[col] / tot
    INT.to_csv(f"{OUT}/interactions_jackson.csv", index=False)
    print(INT.round(3).to_string(index=False))
    del I
except Exception:
    print("!!! interactions failed:\n" + traceback.format_exc(), flush=True)
    FAILED.append("interactions")
tick("interactions")

## Check: grouped permutation importance (drop in macro-F1)

In [ ]:
PERM = []
for src, t in (("Jackson", "Jackson"), ("Danenberg", "Danenberg"), ("Jackson", "Danenberg")):
    X_ = CO[t]["XS"]["nbr"][SAMPLE[t]].copy()
    y_ = CO[t]["y"]["coarse"][SAMPLE[t]]
    base, base_c = macro_f1(y_, MODELS[src].predict(X_))
    for gname in GROUPS:
        cols = np.flatnonzero(GROUP == gname)
        drops = []
        for r_ in range(3):
            Xp = X_.copy()
            perm = np.random.default_rng(SEED + r_).permutation(len(Xp))
            Xp[:, cols] = X_[perm][:, cols]
            f1, f1c = macro_f1(y_, MODELS[src].predict(Xp))
            drops.append((base - f1, base_c - f1c))
        PERM.append(dict(train=src, test=t, group=gname, drop_macro_f1=float(np.mean([d[0] for d in drops])),
                         **{f"drop_{c}": float(np.mean([d[1][k] for d in drops])) for k, c in enumerate(COARSE)}))
PERMD = pd.DataFrame(PERM)
PERMD.to_csv(f"{OUT}/permutation_by_group.csv", index=False)
with pd.option_context("display.width", 250):
    print(PERMD.round(3).to_string(index=False))
tick("permutation check")

## Imputation drivers: SMA (Jackson vs Danenberg) and H3K27me3 (Jackson; Keren with all channels)
XGBoost regressors on the cell's own channels (cohort-scaled; no neighbours), TreeSHAP on 20,000 test cells.

In [ ]:
XREG = dict(XGB, objective="reg:squarederror")
IMP = []

def reg_shap(tag, X, y, fit_rows, test_rows, names, cohort):
    mdl = xgb.XGBRegressor(**XREG).fit(X[fit_rows], y[fit_rows])
    pr = mdl.predict(X[test_rows])
    r2 = 1 - np.sum((y[test_rows] - pr) ** 2) / np.sum((y[test_rows] - y[test_rows].mean()) ** 2)
    sub = np.sort(np.random.default_rng(SEED + 2).choice(test_rows, min(N_SHAP, len(test_rows)), replace=False))
    V = mdl.get_booster().predict(xgb.DMatrix(X[sub]), pred_contribs=True)[:, :-1]
    a = np.abs(V).mean(0)
    for j, f in enumerate(names):
        IMP.append(dict(model=tag, cohort=cohort, feature=f, mean_abs_shap=float(a[j]), share=float(a[j] / a.sum()), r2_test=float(r2)))
    return mdl

try:
    k_sma = BASE.index("sma")
    names = [LABEL[c] for i, c in enumerate(BASE) if i != k_sma]
    cols = [i for i in range(len(BASE)) if i != k_sma]
    M_SMA = {}
    for co in ("Jackson", "Danenberg"):
        C = CO[co]
        X, y = C["XS"]["cohort"][:, cols], C["XS"]["cohort"][:, k_sma]
        tr = cells_of(C, IMG[co]["train"])
        tr = np.sort(np.random.default_rng(SEED).choice(tr, min(N_FIT, len(tr)), replace=False))
        M_SMA[co] = reg_shap(f"SMA, {co}-trained", X, y, tr, cells_of(C, IMG[co]["test"]), names, co)
    # transfer: the Jackson SMA model explained on Danenberg cells
    Xd, yd = CD["XS"]["cohort"][:, cols], CD["XS"]["cohort"][:, k_sma]
    te = cells_of(CD, IMG["Danenberg"]["test"])
    sub = np.sort(np.random.default_rng(SEED + 3).choice(te, min(N_SHAP, len(te)), replace=False))
    V = M_SMA["Jackson"].get_booster().predict(xgb.DMatrix(Xd[sub]), pred_contribs=True)[:, :-1]
    pr = M_SMA["Jackson"].predict(Xd[te])
    r = np.corrcoef(pr, yd[te])[0, 1]
    a = np.abs(V).mean(0)
    for j, f in enumerate(names):
        IMP.append(dict(model="SMA, Jackson-trained, on Danenberg", cohort="Danenberg", feature=f, mean_abs_shap=float(a[j]),
                        share=float(a[j] / a.sum()), r2_test=float(np.sign(r) * r * r)))
    tick("SMA")
except Exception:
    print("!!! SMA block failed:\n" + traceback.format_exc(), flush=True)
    FAILED.append("SMA")

try:
    jc = pd.read_parquet(P["Jackson"]["cells"], columns=["image_name", "object_number", "histone_h3_trimethylate"])
    jc = jc.set_index(["image_name", "object_number"]).histone_h3_trimethylate
    yk = jc.reindex(pd.MultiIndex.from_arrays([CJ["img_names"][np.repeat(np.arange(CJ["n_img"]), np.diff(CJ["starts"]))],
                                               CJ["obj"]])).values.astype(np.float64)
    yk = np.arcsinh(yk / COF)
    ok = np.isfinite(yk)
    tr = cells_of(CJ, IMG["Jackson"]["train"]); tr = tr[ok[tr]]
    tr = np.sort(np.random.default_rng(SEED).choice(tr, min(N_FIT, len(tr)), replace=False))
    te = cells_of(CJ, IMG["Jackson"]["test"]); te = te[ok[te]]
    reg_shap("H3K27me3, Jackson (14 shared channels + area)", CJ["XS"]["cohort"], yk, tr, te, [LABEL[c] for c in BASE], "Jackson")
    tick("H3K27me3 Jackson")
except Exception:
    print("!!! H3K27me3 Jackson failed:\n" + traceback.format_exc(), flush=True)
    FAILED.append("H3K27me3 Jackson")

try:
    kp = [p for p in g("cells.parquet") if "published__h3k27me3" in pq.read_schema(p).names]
    assert len(kp) == 1, kp
    kc = pd.read_parquet(kp[0])
    skip = {"area", "area_um2", "area_eroded", "x", "y", "x_um", "y_um", "h3k27me3", "h3k9ac"}
    kch = [c for c in kc.columns if kc[c].dtype.kind == "f" and c not in skip and not c.startswith(("eroded__", "published__"))]
    Xk = harm(np.c_[np.arcsinh(kc[kch].values.astype(np.float64) / 1.0), np.log(np.maximum(kc.area.values, 1))])
    yk = np.arcsinh(kc.h3k27me3.values.astype(np.float64) / 1.0)
    pats = kc.patient_id.astype(str).values
    up = np.unique(pats)
    test_p = set(np.random.default_rng(SEED).choice(up, len(up) // 5, replace=False))
    is_te = np.isin(pats, list(test_p))
    reg_shap("H3K27me3, Keren (all 34 non-histone-mark channels + area)", Xk, yk, np.flatnonzero(~is_te),
             np.flatnonzero(is_te), kch + ["area"], "Keren")
    tick("H3K27me3 Keren")
except Exception:
    print("!!! H3K27me3 Keren failed:\n" + traceback.format_exc(), flush=True)
    FAILED.append("H3K27me3 Keren")

IMPD = pd.DataFrame(IMP)
IMPD.to_csv(f"{OUT}/imputation_shap.csv", index=False)
with pd.option_context("display.width", 250):
    for mname, d in IMPD.groupby("model", sort=False):
        d = d.sort_values("share", ascending=False)
        print(f"--- {mname} (test R² {d.r2_test.iloc[0]:.3f})")
        print(", ".join(f"{r.feature} {r.share:.2f}" for r in d.head(8).itertuples()))

## Figure and summary

In [ ]:
try:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
    for ax, (src, t) in zip(axes, (("Jackson", "Jackson"), ("Danenberg", "Danenberg"), ("Jackson", "Danenberg"))):
        M = GRP[(GRP.train == src) & (GRP.test == t)].pivot_table(index="group", columns="cls", values="share").reindex(GROUPS)[COARSE]
        im = ax.imshow(M.values, cmap="Blues", vmin=0, vmax=max(0.5, float(np.nanmax(M.values))), aspect="auto")
        ax.set_xticks(range(len(COARSE)))
        ax.set_xticklabels(COARSE, rotation=40, ha="right", fontsize=8)
        ax.set_yticks(range(len(GROUPS)))
        ax.set_yticklabels(GROUPS, fontsize=8)
        ax.set_xlabel(f"True class ({t} test cells, model trained on {src})", fontsize=8)
        ax.set_ylabel("Feature group", fontsize=8)
        for i in range(M.shape[0]):
            for j in range(M.shape[1]):
                v = M.values[i, j]
                if np.isfinite(v) and v >= 0.05:
                    ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6.5, color="white" if v > 0.3 else "black")
    fig.colorbar(im, ax=axes, fraction=0.02, pad=0.01, label="Share of |SHAP| for the true class")
    fig.savefig(f"{OUT}/fig_shap_groups.png", dpi=200, bbox_inches="tight")
    plt.close(fig)
except Exception:
    print("!!! figure failed:\n" + traceback.format_exc(), flush=True)
    FAILED.append("figure")
json.dump(dict(smoke=SMOKE, minutes=(time.time() - T0) / 60, failed=FAILED, xgboost=xgb.__version__,
               scores=SC.round(4).to_dict(orient="records"), driver_agreement=AGREE.round(4).to_dict(orient="records")),
          open(f"{OUT}/summary.json", "w"), indent=1)
print("failed blocks:", FAILED if FAILED else "none")
tick("done")